# CLEF 2026 Humour Project - Colab GPU Pipeline

This notebook runs GPU-heavy tasks on Colab using profile-based commands.

Note: The notebook is shell-magic free and uses portable Python command execution, so it works in both Google Colab and local Jupyter environments.

In [1]:
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Not running in Google Colab; skipping drive mount.")


def run_cmd(cmd, cwd=None):
    """Run a command portably and surface useful error context on failure."""
    cmd_str = " ".join(shlex.quote(str(part)) for part in cmd)
    print("$", cmd_str)

    completed = subprocess.run(
        cmd,
        cwd=cwd,
        text=True,
        capture_output=True,
    )

    if completed.stdout:
        print(completed.stdout, end="")
    if completed.returncode != 0:
        if completed.stderr:
            print("\n[stderr]\n" + completed.stderr)
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {cmd_str}")


def find_repo_root(start_dir: Path) -> Path:
    """Find project root by looking for standard repo markers."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").exists():
            return candidate
    return start_dir


REPO_DIR = str(find_repo_root(Path.cwd()))
print("Repo dir:", REPO_DIR)

venv_python = Path(REPO_DIR) / "venv" / "Scripts" / "python.exe"
if (not IN_COLAB) and venv_python.exists():
    PYTHON_EXE = str(venv_python)
    print("Using project venv Python:", PYTHON_EXE)
else:
    PYTHON_EXE = sys.executable
    print("Using kernel Python:", PYTHON_EXE)

Not running in Google Colab; skipping drive mount.
Repo dir: c:\Users\badsh\OneDrive\Desktop\CLEF_2026_Humour_Project
Using project venv Python: c:\Users\badsh\OneDrive\Desktop\CLEF_2026_Humour_Project\venv\Scripts\python.exe


In [3]:
REPO_URL = 'https://github.com/Badshah1508/Humour-Aware-Information-Retrieval-Pun-Translation-CLEF-2026-.git'

if IN_COLAB:
    REPO_DIR = '/content/CLEF_2026_Humour_Project'
    if not os.path.exists(REPO_DIR):
        run_cmd(["git", "clone", REPO_URL, REPO_DIR])
    else:
        run_cmd(["git", "pull"], cwd=REPO_DIR)
else:
    print('Using local repository directory.')

os.chdir(REPO_DIR)
print('Current working directory:', os.getcwd())

Using local repository directory.
Current working directory: c:\Users\badsh\OneDrive\Desktop\CLEF_2026_Humour_Project


In [4]:
run_cmd([PYTHON_EXE, "-V"])
run_cmd([PYTHON_EXE, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run_cmd([PYTHON_EXE, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

$ 'c:\Users\badsh\OneDrive\Desktop\CLEF_2026_Humour_Project\venv\Scripts\python.exe' -V
Python 3.13.5
$ 'c:\Users\badsh\OneDrive\Desktop\CLEF_2026_Humour_Project\venv\Scripts\python.exe' -m pip install -q --upgrade pip
$ 'c:\Users\badsh\OneDrive\Desktop\CLEF_2026_Humour_Project\venv\Scripts\python.exe' -m pip install -q -r requirements.txt


In [4]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: False


In [5]:
# Fast iteration profile
profile_env = {
    "CE_EPOCHS": "2",
    "CE_BATCH_SIZE": "8",
    "CE_EVAL_STEPS": "1000",
    "CE_MAX_HARD_NEGS": "10",
    "CE_MAX_RANDOM_NEGS": "4",
    "CE_NUM_WORKERS": "2",
    "CE_PIN_MEMORY": "auto",
    "DENSE_MODEL_NAME": "sentence-transformers/all-MiniLM-L6-v2",
    "DENSE_BATCH_SIZE": "32",
    "DENSE_TOP_K": "50",
    "DENSE_EXPERIMENT_TOP_K": "50",
    "CROSS_ENCODER_CANDIDATES": "20",
}

os.environ.update(profile_env)
print("Profile environment variables set:")
for key in profile_env:
    print(f"{key}={os.environ[key]}")

Profile environment variables set:
CE_EPOCHS=2
CE_BATCH_SIZE=8
CE_EVAL_STEPS=1000
CE_MAX_HARD_NEGS=10
CE_MAX_RANDOM_NEGS=4
CE_NUM_WORKERS=2
CE_PIN_MEMORY=auto
DENSE_MODEL_NAME=sentence-transformers/all-MiniLM-L6-v2
DENSE_BATCH_SIZE=32
DENSE_TOP_K=50
DENSE_EXPERIMENT_TOP_K=50
CROSS_ENCODER_CANDIDATES=20


In [ ]:
# Run fast dense retrieval
run_cmd([PYTHON_EXE, "-m", "src.retrieval.run_dense"])

$ 'c:\Users\badsh\OneDrive\Desktop\CLEF_2026_Humour_Project\venv\Scripts\python.exe' -m src.retrieval.run_dense


In [ ]:
# Run fast cross-encoder training
run_cmd([PYTHON_EXE, "-m", "src.retrieval.train_cross_encoder"])

In [ ]:
# Run reranking
run_cmd([PYTHON_EXE, "-m", "src.retrieval.run_cross_encoder"])

In [ ]:
# Optional: save outputs back to Drive (Colab only)
if IN_COLAB:
    out_dir = Path('/content/drive/MyDrive/CLEF_2026_outputs')
    out_dir.mkdir(parents=True, exist_ok=True)

    results_src = Path('results')
    model_src = Path('models/reranker/cross_encoder_finetuned')

    if results_src.exists():
        shutil.copytree(results_src, out_dir / 'results', dirs_exist_ok=True)
    if model_src.exists():
        shutil.copytree(model_src, out_dir / 'cross_encoder_finetuned', dirs_exist_ok=True)

    print(f'Saved outputs to {out_dir}')
else:
    print('Not in Colab; outputs are already available in local results/ and models/.')

Not in Colab; outputs are already available in local results/ and models/.
